In [1]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import tensorflow as tf 
from sklearn.preprocessing import normalize

2025-01-13 15:01:51.446151: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


# Prepare the data

In [35]:
game_folder = "data/game_data_with_tags.csv"
time_folder = "data/time_data.csv"

game_data = pd.read_csv(game_folder)
time_data = pd.read_csv(time_folder)

In [79]:
game_time_sum = time_data.groupby('game_id')['user_id'].count()
game_time_sum = game_time_sum.reset_index()
game_time_sum.columns = ['game_id', 'users']
game_time_sum = game_time_sum.sort_values(by='users', ascending=False)
# get ids of games with at least 50 users, and filter the time_data
game_time_sum = game_time_sum[game_time_sum['users'] >= 50]


# get the game ids
game_ids = game_time_sum['game_id']


# filter the game_data
game_data = game_data[game_data['game_id'].isin(game_ids)]

# get rid of games that are called unknown_game in the game_data
game_data = game_data[game_data['game_name'] != 'unknown game']

len(game_data)

7066

In [80]:
uniq_tags = []
for row in game_data.itertuples():
    if row.tag_0: uniq_tags.append(row.tag_0)
    if row.tag_1: uniq_tags.append(row.tag_1)
    if row.tag_2: uniq_tags.append(row.tag_2)
    if row.tag_3: uniq_tags.append(row.tag_3)
    if row.tag_4: uniq_tags.append(row.tag_4)
uniq_tags = list(set(uniq_tags))
# remove nan
uniq_tags = [tag for tag in uniq_tags if tag == tag]

# create encoding of tags to integers
tags2int = {k: v for v, k in enumerate(uniq_tags)}

len(uniq_tags)

412

In [81]:
# find rows where tag_0 is nan, remove them (that means the game has no tags)
game_data = game_data[game_data["tag_0"] == game_data["tag_0"]]

# remove all time_data rows where the game_id is not in game_data
time_data = time_data[time_data['game_id'].isin(game_data['game_id'])]

In [82]:
game_tag_vectors = {}
for row in game_data.itertuples():
    game_tag_vectors[row.game_id] = [0] * len(uniq_tags)
    for i, tag in enumerate(uniq_tags):
        if row.tag_0 == tag: game_tag_vectors[row.game_id][i] = 1
        if row.tag_1 == tag: game_tag_vectors[row.game_id][i] = 1
        if row.tag_2 == tag: game_tag_vectors[row.game_id][i] = 1
        if row.tag_3 == tag: game_tag_vectors[row.game_id][i] = 1
        if row.tag_4 == tag: game_tag_vectors[row.game_id][i] = 1

for key in game_tag_vectors:
    game_tag_vectors[key] = normalize([game_tag_vectors[key]], axis=1, norm='l2')[0]

In [83]:
time_data = time_data.sample(frac=1, random_state=42)
train_data = time_data[:int(0.9 * len(time_data))]
val_data = time_data[int(0.9 * len(time_data)):]

In [84]:
uniq_users = list(set(train_data['user_id']))

In [85]:
# make a vector for each game
game_tag_vectors = {}
for row in game_data.itertuples():
    game_tag_vectors[row.game_id] = [0] * len(uniq_tags)
    for i, tag in enumerate(uniq_tags):
        if row.tag_0 == tag: game_tag_vectors[row.game_id][i] = 1
        if row.tag_1 == tag: game_tag_vectors[row.game_id][i] = 1
        if row.tag_2 == tag: game_tag_vectors[row.game_id][i] = 1
        if row.tag_3 == tag: game_tag_vectors[row.game_id][i] = 1
        if row.tag_4 == tag: game_tag_vectors[row.game_id][i] = 1

# normalize the vectors, keep the dictionary
for key in game_tag_vectors:
    game_tag_vectors[key] = normalize([game_tag_vectors[key]], axis=1, norm='l2')[0]

time_data = time_data.sample(frac=1, random_state=42)
train_data = time_data[:int(0.9 * len(time_data))]
val_data = time_data[int(0.9 * len(time_data)):]

uniq_users = list(set(train_data['user_id']))

# Evaluate the model

In [13]:
rec_tally = 0
for iter, user in tqdm(enumerate(uniq_users), total=len(uniq_users)):
    user_data = train_data[train_data['user_id'] == user]

    played_tags = {}

    for row in user_data.itertuples():
        game_tags = game_data.loc[game_data["game_id"] == row.game_id, "tag_0":"tag_4"].values[0]
        for tag in game_tags:
            # check if tag is nan
            if tag != tag:
                continue
            played_tags[tag] = played_tags.get(tag, 0) + row.playtime

    user_tags_vector = np.zeros(len(uniq_tags))
    user_id = 1
    for item in played_tags:
        user_tags_vector[tags2int[item]] = played_tags[item]

    user_tags_vector = normalize([user_tags_vector], norm='l2')[0]

    # compute pairwise cosine similarity
    cosine_similarities = {}

    for key in game_tag_vectors:
        game_vector = game_tag_vectors[key]
        cosine_similarities[key] = np.dot(user_tags_vector, game_vector)

    # get rid of games that the user has already played
    for game_id in user_data["game_id"]:
        cosine_similarities.pop(game_id, None)

    recommendations = sorted(cosine_similarities.items(),
                            key=lambda x: x[1], reverse=True)[:20]
    recommendations = [x[0] for x in recommendations]


    # check how many of the recommendations are in the validation set
    val_games = set(val_data[val_data['user_id'] == user]['game_id'])
    rec_tally += len(set(recommendations).intersection(val_games))

print("Total recommendations in validation set:", rec_tally)
print("Total users:", len(uniq_users))
print("Recommendations per user:", rec_tally / len(uniq_users))

  0%|          | 0/6571 [00:00<?, ?it/s]

Total recommendations in validation set: 1915
Total users: 6571
Recommendations per user: 0.2914320499162989


# Generate game recommendations for random user

In [101]:
# take a random user
user_id = np.random.choice(uniq_users)
user_time_data = time_data[time_data['user_id'] == user_id]
user_time_data.head(10)

,user_id,game_id,playtime
860144,1401,2463,156
860117,1401,8433,326
860003,1401,4966,9
860092,1401,17739,275
860174,1401,2889,270
860149,1401,8727,232
860153,1401,4783,134
860027,1401,1334,113
859967,1401,4483,1
860124,1401,2337,728


In [102]:
played_tags = {}

for row in user_time_data.itertuples():
    game_tags = game_data.loc[game_data["game_id"] == row.game_id, "tag_0":"tag_4"].values[0]
    for tag in game_tags:
        # check if tag is nan
        if tag != tag:
            continue
        played_tags[tag] = played_tags.get(tag, 0) + row.playtime

In [103]:
user_tags_vector = np.zeros(len(uniq_tags))
for item in played_tags:
    user_tags_vector[tags2int[item]] = played_tags[item]

user_tags_vector = normalize([user_tags_vector], norm='l2')
user_tags_tensor = tf.convert_to_tensor(user_tags_vector, dtype=tf.float32)

In [104]:
# compute pairwise cosine similarity
cosine_similarities = {}

for key in game_tag_vectors:
    game_vector = game_tag_vectors[key]
    game_vector_tensor = tf.convert_to_tensor([game_vector], dtype=tf.float32)
    cosine_similarities[key] = tf.keras.losses.cosine_similarity(user_tags_tensor, game_vector_tensor).numpy()[0]

# get rid of games that the user has already played
for game_id in user_time_data["game_id"]:
    cosine_similarities.pop(game_id, None)

In [105]:
# extract 10 most similar games, and their ids
recommendations = sorted(cosine_similarities.items(), key=lambda x: x[1], reverse=False)[:10]

In [106]:
# get the game names and tags
game_names = []
tags = []
for game_id, _ in recommendations:
    game_row = game_data.loc[game_data["game_id"] == game_id]
    game_names.append(game_row["game_name"].values[0])
    game_tags = []
    for i in range(5):
        if game_row[f"tag_{i}"].values[0]: game_tags.append(game_row[f"tag_{i}"].values[0])
    tags.append(game_tags)

# get 10 most played games by the user
most_played = user_time_data.sort_values(by='playtime', ascending=False)[:10]
most_played_games = []
for row in most_played.itertuples():
    game_row = game_data.loc[game_data["game_id"] == row.game_id]
    most_played_games.append((game_row["game_name"].values[0], row.playtime // 60))

print("Most played games:")
for item in most_played_games:
    print(item[0], "-", item[1], "hours")

print("\nRecommendations:")
for item in zip(game_names, tags):
    print(item)

Most played games:
Warframe - 735 hours
Dwarf Fortress - 75 hours
Project Zomboid - 75 hours
Remnant II - 54 hours
Starfield - 51 hours
Borderlands 3 - 35 hours
No Man's Sky - 35 hours
THRONE AND LIBERTY - 28 hours
Cyberpunk 2077 - 27 hours
Need for Speed™ Heat  - 26 hours

Recommendations:
("Tom Clancy's The Division", ['Open World', 'Looter Shooter', 'Multiplayer', 'Third-Person Shooter', 'Action'])
('OUTRIDERS', ['Looter Shooter', 'RPG', 'Co-op', 'Third-Person Shooter', 'Action'])
('Hazard Ops', ['Free to Play', 'Action', 'Zombies', 'Multiplayer', 'Third-Person Shooter'])
('BorderZone', ['RPG', 'Action', 'Action RPG', nan, nan])
('Wave of Darkness', ['Action', 'RPG', 'Indie', 'Action RPG', nan])
('GunZ 2: The Second Duel', ['Free to Play', 'Action', 'Multiplayer', 'Third-Person Shooter', 'Hack and Slash'])
('Rogue Company', ['Free to Play', 'Multiplayer', 'Shooter', 'Third-Person Shooter', 'Action'])
('tModLoader', ['Adventure', 'Free to Play', 'Action', 'Indie', 'RPG'])
('AirBuccan